## This notebook creates a table with monthly seascape class and cruise metadata, and renders data visualizations and seascape maps.

In [1]:
# Load libraries
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import requests
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import BoundaryNorm
from scipy.spatial import KDTree
from scipy.stats import mode as sp_mode

## Load Argo data already processed by Jen. SKIP if querying Argo data from ERDDAP

In [ ]:
# Define the path to the data file
argo_file = Path('Gulf_BGCArgo_13May2026.csv').expanduser()

# Load the data into a pandas DataFrame
argo_df = pd.read_csv(argo_file)

# Display the first few rows of the DataFrame to verify it loaded correctly
argo_df.head()

# Print few rows of the DataFrame to verify it loaded correctly
print(argo_df.tail(1))

## SKIP if querying Argo data from ERDDAP

In [ ]:
# Convert JULD (Julian Day) to datetime
# JULD is typically days since 1950-01-01 in Argo data
argo_df['datetime'] = pd.to_datetime(argo_df['JULD'], unit='D', origin='1950-01-01')

# Display the result
print(argo_df.tail(1))

## Query Argo data from ERDDAP server. SKIP IF DATA ALREADY DOWNLOADED FROM ERDDAP

In [3]:
import os
import requests

# Define the list of Argo float IDs
float_ids = [4903622, 4903624, 4903625, 7901009, 2904010,
             2904011, 6999992, 4903904, 4903910, 4903911,
             7902327, 3902693, 1902800, 5907206]

# Destination folder for downloaded files
dest_folder = 'data_argo/argo_sprof/'
os.makedirs(dest_folder, exist_ok=True)

print("--- Starting Argo Sprof Data Download ---")

# Download the latest Sprof file for each float
for float_id in float_ids:
    url = f"https://data-argo.ifremer.fr/dac/aoml/{float_id}/{float_id}_Sprof.nc"
    dest_file = os.path.join(dest_folder, f"{float_id}_Sprof.nc")

    print(f"Downloading {float_id}_Sprof.nc...")

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() 

        with open(dest_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
        print(f"Success: Latest file downloaded for float {float_id}")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading float {float_id}: {e}")

print("=== All downloads completed ===")

--- Starting Argo Sprof Data Download ---
Success: Latest file downloaded for float 4903622
Success: Latest file downloaded for float 4903624
Success: Latest file downloaded for float 4903625
Success: Latest file downloaded for float 7901009
Success: Latest file downloaded for float 2904010
Success: Latest file downloaded for float 2904011
Success: Latest file downloaded for float 6999992
Success: Latest file downloaded for float 4903904
Success: Latest file downloaded for float 4903910
Success: Latest file downloaded for float 4903911
Success: Latest file downloaded for float 7902327
Success: Latest file downloaded for float 3902693
Success: Latest file downloaded for float 1902800
Success: Latest file downloaded for float 5907206
=== All downloads completed ===


## Process Argo data downloaded from ERDDAP. 

In [4]:
import os
import netCDF4
import gsw
import pandas as pd
import numpy as np

argo_processed_output_path = 'argo_sprof_processed_data.csv'

# Extract profile data from a float's NetCDF file
def process_float(file_path, wmo_id, juld_origin="1950-01-01"):
    nc_file = netCDF4.Dataset(file_path, 'r')
    cycle_numbers = nc_file.variables['CYCLE_NUMBER'][:]
    directions = nc_file.variables['DIRECTION'][:]
    data_mode = nc_file.variables['DATA_MODE'][:].tobytes().decode('utf-8').strip() if 'DATA_MODE' in nc_file.variables else 'U'
    
    profiles = []
    
    for i in range(len(cycle_numbers)):
        cycle_number = cycle_numbers[i]
        direction = directions[i]
        
        # Get index for current cycle and direction
        profile_indices = np.where((nc_file.variables['CYCLE_NUMBER'][:] == cycle_number) & 
                                   (nc_file.variables['DIRECTION'][:] == direction))[0]
                                   
        if len(profile_indices) == 0:
            continue
            
        p_idx = profile_indices[0]
        
        # Extract base variables
        lat = nc_file.variables['LATITUDE'][p_idx]
        lon = nc_file.variables['LONGITUDE'][p_idx]
        juld = nc_file.variables['JULD'][p_idx]
        profile_datetime = pd.to_datetime(juld, unit='D', origin=juld_origin)

        pressure = nc_file.variables['PRES_ADJUSTED'][p_idx, :].filled(np.nan)
        temperature = nc_file.variables['TEMP_ADJUSTED'][p_idx, :].filled(np.nan)
        salinity = nc_file.variables['PSAL_ADJUSTED'][p_idx, :].filled(np.nan)
        
        # Helper function to safely extract BGC variables (fills with NaN if missing)
        def get_var(var_name):
            if var_name in nc_file.variables:
                return nc_file.variables[var_name][p_idx, :].filled(np.nan)
            return np.full_like(pressure, np.nan)
        
        # Only process if essential BGC variables are present
        if 'CHLA_ADJUSTED' in nc_file.variables and 'NITRATE_ADJUSTED' in nc_file.variables:
             chla = get_var('CHLA_ADJUSTED')
             nitrate = get_var('NITRATE_ADJUSTED')
             
             # Extract new requested parameters (both raw and adjusted)
             doxy = get_var('DOXY')
             doxy_adj = get_var('DOXY_ADJUSTED')
             
             bbp700 = get_var('BBP700')
             bbp700_adj = get_var('BBP700_ADJUSTED')
             
             ph = get_var('PH_IN_SITU_TOTAL')
             ph_adj = get_var('PH_IN_SITU_TOTAL_ADJUSTED')
             
             cdom = get_var('CDOM')
             cdom_adj = get_var('CDOM_ADJUSTED')
             
             profile_df = pd.DataFrame({
                'WMOID': wmo_id,
                'CYCLE_NUMBER': cycle_number,
                'direction': direction,
                'profile_mode': data_mode,
                'JULD': juld,
                'datetime': profile_datetime,
                'LATITUDE': lat,
                'LONGITUDE': lon,
                'pressure': pressure,
                'temperature': temperature,
                'salinity': salinity,
                'chla': chla,
                'nitrate': nitrate,
                'doxy': doxy,
                'doxy_adjusted': doxy_adj,
                'bbp700': bbp700,
                'bbp700_adjusted': bbp700_adj,
                'ph_in_situ_total': ph,
                'ph_in_situ_total_adjusted': ph_adj,
                'cdom': cdom,
                'cdom_adjusted': cdom_adj
             })
             
             # Calculate Density
             latitude_for_gsw = profile_df['LATITUDE'].iloc[0] if not profile_df['LATITUDE'].empty else 0
             salinity_absolute = gsw.SA_from_SP(profile_df['salinity'].values, profile_df['pressure'].values, profile_df['LONGITUDE'].values, latitude_for_gsw)
             conservative_temperature = gsw.CT_from_t(salinity_absolute, profile_df['temperature'].values, profile_df['pressure'].values)
             profile_df['density'] = gsw.rho(salinity_absolute, conservative_temperature, profile_df['pressure'].values)
             
             # Filter out rows with NaN in essential columns
             profile_df = profile_df.dropna(subset=['pressure', 'temperature', 'salinity'])
             profiles.append(profile_df)
             
    nc_file.close()
    return pd.concat(profiles, ignore_index=True) if profiles else None

print("--- Starting Argo Sprof Data Processing ---")

all_processed_dfs = []

# Process each downloaded file
for float_id in float_ids:
    file_path = os.path.join(dest_folder, f"{float_id}_Sprof.nc")
    if os.path.exists(file_path):
        print(f"Processing: {file_path}")
        df = process_float(file_path, str(float_id))
        if df is not None:
             all_processed_dfs.append(df)

# Combine and save
if all_processed_dfs:
    argo_df = pd.concat(all_processed_dfs, ignore_index=True)
    argo_df.to_csv(argo_processed_output_path, index=False)
    print(f"Processed Argo data saved to: {argo_processed_output_path}")
else:
    print("No data processed.")

--- Starting Argo Sprof Data Processing ---
Processing: data_argo/argo_sprof/4903622_Sprof.nc
Processing: data_argo/argo_sprof/4903624_Sprof.nc
Processing: data_argo/argo_sprof/4903625_Sprof.nc
Processing: data_argo/argo_sprof/7901009_Sprof.nc
Processing: data_argo/argo_sprof/2904010_Sprof.nc
Processing: data_argo/argo_sprof/2904011_Sprof.nc
Processing: data_argo/argo_sprof/6999992_Sprof.nc
Processing: data_argo/argo_sprof/4903904_Sprof.nc
Processing: data_argo/argo_sprof/4903910_Sprof.nc
Processing: data_argo/argo_sprof/4903911_Sprof.nc
Processing: data_argo/argo_sprof/7902327_Sprof.nc
Processing: data_argo/argo_sprof/3902693_Sprof.nc
Processing: data_argo/argo_sprof/1902800_Sprof.nc
Processing: data_argo/argo_sprof/5907206_Sprof.nc
Processed Argo data saved to: argo_sprof_processed_data.csv


## Query seascape data

In [ ]:
# Change this variable to 'SEASCAPE_MONTH' or 'SEASCAPE_8DAY' as needed
dataset_type = 'SEASCAPE_MONTH' 

# Define Domain Limits and Dates 
n_lat = 32
s_lat = 10
e_lon = -59
w_lon = -100
start_date = '2021-08-15T12:00:00Z'
end_date = '2026-05-15T12:00:00Z'

# Build the Request for the MONTHLY Product 
print(f"Configuring request for: {dataset_type}")

base_url = f"https://cwcgom.aoml.noaa.gov/thredds/ncss/{dataset_type}/SEASCAPES.nc"

params = {
    'var': ['CLASS', 'P'],
    'north': n_lat,
    'west': w_lon,
    'east': e_lon,
    'south': s_lat,
    'disableProjSubset': 'on',
    'horizStride': 1,
    'time_start': start_date,
    'time_end': end_date,
    'timeStride': 1,
    'addLatLon': 'true',
    'accept': 'netcdf',
}

req = requests.Request('GET', base_url, params=params).prepare()
url = req.url

print("Requesting data from URL:")
print(url)

# Download the Data Manually and Then Open ---
output_filename = f"seascape_argo_data_{dataset_type.lower()}.nc"

try:
    # Download the file using requests
    print(f"\nDownloading data to '{output_filename}'...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()  # This will raise an error for bad status codes
        with open(output_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Download complete.")

    # Open the local file with xarray
    ds = xr.open_dataset(output_filename)
    print("\nSuccessfully loaded dataset from local file:")
    print(ds)

    # Analyze Unique Classes ---
    class_data = ds['CLASS'].values
    unique_classes_with_nan = np.unique(class_data)
    unique_classes = unique_classes_with_nan[~np.isnan(unique_classes_with_nan)]

    print("\nUnique classes present in CLASS:")
    print(unique_classes.astype(int))

except Exception as err:
    print(f"\nAn error occurred: {err}")

## Create date-time vector for seascape data

In [ ]:
# Create TIMEVEC DataFrame.
# Access the time coordinate from the xarray dataset
time_coordinate = ds['time']

# Create a dictionary with the year, month, day, and full datetime
time_data = {
    'year': time_coordinate.dt.year.values,
    'month': time_coordinate.dt.month.values,
    'day': time_coordinate.dt.day.values,
    'datetime': time_coordinate.values
}

# Convert the dictionary into a pandas DataFrame
TIMEVEC = pd.DataFrame(time_data)

print("Created TIMEVEC DataFrame:")
print(TIMEVEC)

###  Map Seascape Data for a Single Time Step as a test

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

# Select the Data for a Single Time Step 
seascape_selected = ds.isel(time=45)

# Find every unique class across the entire dataset to ensure consistency
all_values = ds['CLASS'].values
global_classes = np.unique(all_values[~np.isnan(all_values)]).astype(int)
num_global_classes = len(global_classes)

# Select a base color palette. 
if num_global_classes <= 20:
    colors = plt.get_cmap('tab20b')(np.linspace(0, 1, num_global_classes))
else:
    combined_palette = np.vstack([
        plt.get_cmap('tab20b')(np.linspace(0, 1, 20)),
        plt.get_cmap('tab20c')(np.linspace(0, 1, 20))
    ])
    colors = combined_palette[:num_global_classes]

# Create the global, unchanging colormap and normalization bounds
common_cmap = ListedColormap(colors)
common_bounds = np.append(global_classes, global_classes[-1] + 1) - 0.5
common_norm = BoundaryNorm(common_bounds, ncolors=common_cmap.N)

# --- Set Up the Map ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())

# --- Plot the Seascape Data ---
mesh = ax.pcolormesh(
    seascape_selected['lon'], 
    seascape_selected['lat'], 
    seascape_selected['CLASS'],
    cmap=common_cmap,
    norm=common_norm, # Apply the normalization rule
    transform=ccrs.PlateCarree()
)

# --- Add Map Features ---
ax.add_feature(cfeature.LAND, zorder=1, edgecolor='black', facecolor='tan')
ax.add_feature(cfeature.COASTLINE)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

# --- Add Colorbar and Title ---
cbar = plt.colorbar(mesh, ax=ax, ticks=unique_classes)
cbar.set_label('Seascape Class ID', fontsize=12, weight='bold')

# Get the date from the selected data
seascape_date = pd.to_datetime(seascape_selected.time.values).strftime('%Y-%m-%d')

# Set the title using the extracted date
ax.set_title(f'Seascapes for {seascape_date}', fontsize=16, weight='bold')

plt.show()

### Define the parameters and prepare the data structures needed for the analysis.

In [ ]:
# --- Parameters ---
N_PIXELS = 3  # Radius in pixels to search around each station
PIX_RESOLUTION_DEG = 0.0416 # Approximate resolution of 4-km seascapes in degrees
SEARCH_RADIUS_DEG = N_PIXELS * PIX_RESOLUTION_DEG

# Prepare the seascape grid coordinates for efficient searching
# Flatten the lat/lon grid into a list of (lat, lon) points
lon_grid, lat_grid = np.meshgrid(ds['lon'].values, ds['lat'].values)
grid_points = np.vstack([lat_grid.ravel(), lon_grid.ravel()]).T

# Create a KD-Tree for fast nearest-neighbor searches. 
kdtree = KDTree(grid_points)

### USE THE NEXT CELL FOR MONTHLY SEASCAPE EXTRACTIONS

In [ ]:
# FOR MONTHLY EXTRACTIONS: This loop iterates through each month, creates the map,
#  and performs the extraction and analysis.

# List to store the final results
all_sites_summary_monthly = []

# Loop through each time step (month) in the seascape data
for i, t_row in TIMEVEC.iterrows():
    current_year = t_row['year']
    current_month = t_row['month']

    # --- Find corresponding argo profiles for this month and year ---
    monthly_argo_df = argo_df.loc[
        (argo_df['datetime'].dt.year == current_year) &
        (argo_df['datetime'].dt.month == current_month)
    ].copy()

    if monthly_argo_df.empty:
        print(f"No argo profiles for {current_year}-{current_month:02d}. Skipping.")
        continue

    print(f"Processing {len(monthly_argo_df)} argo profiles for {current_year}-{current_month:02d}...")

    seascape_monthly = ds.isel(time=i)
    
    # # --- A. Create the Map ---
    # if not monthly_argo_df.empty:
    #     fig = plt.figure(figsize=(10, 8))
    #     ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    #     ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())

    #     # Use BoundaryNorm for correct categorical coloring
    #     if unique_classes.size > 0:
    #         common_bounds = np.append(unique_classes, unique_classes[-1] + 1) - 0.5
    #         common_norm = BoundaryNorm(common_bounds, ncolors=common_cmap.N)
    #         mesh = ax.pcolormesh(
    #             seascape_monthly['lon'], seascape_monthly['lat'], seascape_monthly['CLASS'],
    #             cmap=common_cmap, norm=common_norm, transform=ccrs.PlateCarree()
    #         )
    #         cbar = plt.colorbar(mesh, ax=ax, ticks=unique_classes, pad=0.05, aspect=25)
    #         cbar.set_label('Seascape Class ID', fontsize=12)


    # mesh = ax.pcolormesh(
    #     seascape_monthly['lon'], seascape_monthly['lat'], seascape_monthly['CLASS'],
    #     cmap=common_cmap, norm=common_norm, transform=ccrs.PlateCarree()
    # )

    # ax.add_feature(cfeature.LAND, zorder=1, edgecolor='black', facecolor='tan')
    # ax.add_feature(cfeature.COASTLINE)
    # ax.gridlines(draw_labels=True, linestyle='--')
    # ax.scatter(
    #     monthly_argo_df['LONGITUDE'], monthly_argo_df['LATITUDE'],
    #     s=50, c='black', marker='*', transform=ccrs.PlateCarree(),
    #     label='Cruise Stations'
    # )
    # ax.set_title(f"Seascapes: {current_year}-{current_month:02d}", fontsize=16)
    # plt.savefig(f"{current_year}-{current_month:02d}_map.png", dpi=150, bbox_inches='tight')
    # plt.close(fig)

    # --- B. Extract Surrounding Seascape Pixels ---
    station_points = monthly_argo_df[['LATITUDE', 'LONGITUDE']].values
    nearby_indices_list = kdtree.query_ball_point(station_points, r=SEARCH_RADIUS_DEG)
    monthly_argo_df['nearby_grid_indices'] = nearby_indices_list
    
    # --- C. Calculate Modal Seascape Class ---
    site_summaries = []
    for idx, station in monthly_argo_df.iterrows():
        flat_indices = station['nearby_grid_indices']
        
        # Check if any grid cells were found nearby
        if not flat_indices:
            # Case 1: No grid cells within search radius. Assign NaN and record.
            modal_class = np.nan
            mean_prob_of_modal = np.nan
        else:
            # Grid cells were found, now check if they are valid (not NaN)
            rows, cols = np.unravel_index(flat_indices, seascape_monthly['CLASS'].shape)
            nearby_classes = seascape_monthly['CLASS'].values[rows, cols]
            nearby_probs = seascape_monthly['P'].values[rows, cols]
            
            valid_mask = ~np.isnan(nearby_classes)
            valid_classes = nearby_classes[valid_mask]
            
            if valid_classes.size > 0:
                # Case 2: Valid (non-NaN) grid cells exist. Compute mode.
                modal_class = sp_mode(valid_classes, keepdims=False).mode
                valid_probs = nearby_probs[valid_mask]
                probs_of_modal_class = valid_probs[valid_classes == modal_class]
                mean_prob_of_modal = np.mean(probs_of_modal_class)
            else:
                # Case 3: All nearby grid cells were NaN. Assign NaN and record.
                modal_class = np.nan
                mean_prob_of_modal = np.nan

        # Format the window dates into a clean string identifier
        window_start = pd.Timestamp(year=current_year, month=current_month, day=1)
        window_end = window_start + pd.offsets.MonthEnd(0)
        window_start_str = window_start.strftime('%Y-%m-%d')
        window_end_str = window_end.strftime('%Y-%m-%d')
        time_window_label = f"{window_start_str}_to_{window_end_str}"

        # Append the result for the station, covering all cases to ensure no station is skipped
        site_summaries.append({
            'CYCLE_NUMBER': station['CYCLE_NUMBER'], 
            'WMOID': station['WMOID'],
            'LATITUDE': station['LATITUDE'],
            'LONGITUDE': station['LONGITUDE'],
            'JULD': station['JULD'], 
            'datetime': station['datetime'],
            'MONTH_WINDOW': time_window_label, 
            'modal_seascape_class': modal_class, 
            'mean_seascape_prob': mean_prob_of_modal
        })

    all_sites_summary_monthly.extend(site_summaries)

# Convert the final list of results into a pandas DataFrame
conc_class_per_site_monthly = pd.DataFrame(all_sites_summary_monthly)

# Define the output filename
output_filename = "seascape_argo_summary_monthly.tsv"

# Save the DataFrame to a .tsv file
conc_class_per_site_monthly.to_csv(output_filename, sep='\t', index=False)

print("\n--- Analysis Complete ---")
print("Final summary of modal seascape classes for Argo profiles:")
print(conc_class_per_site_monthly.tail(1))
print(f"\nResults saved to '{output_filename}'")

### USE THE NEXT CELL FOR 8-DAY SEASCAPE EXTRACTIONS

In [ ]:
# FOR 8-DAY EXTRACTIONS: Loop through each 8-day time window
# Prepare for Storing Results ---
all_sites_summary_8day = []

# We stop at -1 because each window is defined by a start (i) and end (i+1) time
for i in range(len(ds.time) - 1):
    start_date = ds.time.values[i]
    end_date = ds.time.values[i+1]

    # --- Find corresponding argo data within the 8-day window ---
    argo_in_window_df = argo_df[
        (argo_df['datetime'] >= start_date) & 
        (argo_df['datetime'] < end_date) 
    ].copy()

    print(f"Processing {len(argo_in_window_df)} window starting {pd.to_datetime(start_date).strftime('%Y-%m-%d')}...")

    # Select the seascape data for the current time step
    seascape_8day = ds.isel(time=i)
    
    # # --- A. Create the Map ---
    # if not argo_in_window_df.empty:
    #     fig = plt.figure(figsize=(10, 8))
    #     ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    #     ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())

    #     # Use BoundaryNorm for correct categorical coloring
    #     if unique_classes.size > 0:
    #         common_bounds = np.append(unique_classes, unique_classes[-1] + 1) - 0.5
    #         common_norm = BoundaryNorm(common_bounds, ncolors=common_cmap.N)
    #         mesh = ax.pcolormesh(seascape_8day['lon'], seascape_8day['lat'], seascape_8day['CLASS'], cmap=common_cmap, norm=common_norm, transform=ccrs.PlateCarree())
    #         cbar = plt.colorbar(mesh, ax=ax, ticks=unique_classes, pad=0.05, aspect=25)
    #         cbar.set_label('Seascape Class ID', fontsize=12)

    #     ax.add_feature(cfeature.LAND, zorder=1, edgecolor='black', facecolor='tan')
    #     ax.add_feature(cfeature.COASTLINE)
    #     ax.gridlines(draw_labels=True, linestyle='--')
    #     ax.scatter(argo_in_window_df['LONGITUDE'], argo_in_window_df['LATITUDE'], s=50, c='black', marker='*', transform=ccrs.PlateCarree(), label='Argo Profiles')
    #     start_date_str = pd.to_datetime(start_date).strftime('%Y-%m-%d')
    #     end_date_str = pd.to_datetime(end_date).strftime('%Y-%m-%d')
    #     ax.set_title(f"Seascapes: {start_date_str}_to_{end_date_str}", fontsize=16)
    #     plt.savefig(f"{start_date_str}_to_{end_date_str}_map.png", dpi=150, bbox_inches='tight')
    #     plt.close(fig)

    # --- B. Extract Surrounding Seascape Pixels ---
    station_points = argo_in_window_df[['LATITUDE', 'LONGITUDE']].values
    nearby_indices_list = kdtree.query_ball_point(station_points, r=SEARCH_RADIUS_DEG)
    argo_in_window_df['nearby_grid_indices'] = nearby_indices_list
    
    # --- C. Calculate Modal Seascape Class ---
    site_summaries = []
    for idx, station in argo_in_window_df.iterrows():
        flat_indices = station['nearby_grid_indices']
        
        # Check if any grid cells were found nearby
        if not flat_indices:
            # Case 1: No grid cells within search radius. Assign NaN.
            modal_class = np.nan
            mean_prob_of_modal = np.nan
        else:
            # Grid cells were found, now check if they are valid (not NaN)
            rows, cols = np.unravel_index(flat_indices, seascape_8day['CLASS'].shape)
            nearby_classes = seascape_8day['CLASS'].values[rows, cols]
            nearby_probs = seascape_8day['P'].values[rows, cols]
            
            valid_mask = ~np.isnan(nearby_classes)
            valid_classes = nearby_classes[valid_mask]
            
            if valid_classes.size > 0:
                # Case 2: Valid (non-NaN) grid cells exist. Compute mode.
                modal_class = sp_mode(valid_classes, keepdims=False).mode
                valid_probs = nearby_probs[valid_mask]
                probs_of_modal_class = valid_probs[valid_classes == modal_class]
                mean_prob_of_modal = np.mean(probs_of_modal_class)
            else:
                # Case 3: All nearby grid cells were NaN. Assign NaN.
                modal_class = np.nan
                mean_prob_of_modal = np.nan

        # Format the window dates into a clean string identifier
        window_start_str = pd.to_datetime(start_date).strftime('%Y-%m-%d')
        window_end_str = pd.to_datetime(end_date).strftime('%Y-%m-%d')
        time_window_label = f"{window_start_str}_to_{window_end_str}"

        # Append the result for the station, covering all cases to ensure no station is skipped
        site_summaries.append({
            'CYCLE_NUMBER': station['CYCLE_NUMBER'], 
            'WMOID': station['WMOID'],
            'LATITUDE': station['LATITUDE'],
            'LONGITUDE': station['LONGITUDE'],
            'JULD': station['JULD'], 
            'datetime': station['datetime'],
            '8_DAY_WINDOW': time_window_label, 
            'modal_seascape_class': modal_class, 
            'mean_seascape_prob': mean_prob_of_modal
        })
    all_sites_summary_8day.extend(site_summaries)

# Convert the final list of results into a pandas DataFrame
conc_class_per_site_8day = pd.DataFrame(all_sites_summary_8day)

# Define the output filename
output_filename = "seascape_argo_summary_8day.tsv"

# Save the DataFrame to a .tsv file
conc_class_per_site_8day.to_csv(output_filename, sep='\t', index=False)

print("\n--- 8-Day Analysis Complete ---")
print("Final summary of modal seascape classes for Argo profiles:")
print(conc_class_per_site_8day.tail(1))
print(f"\nResults saved to '{output_filename}'")

### Check a specific seascape

In [ ]:
windows_23 = conc_class_per_site_8day.loc[
    conc_class_per_site_8day['modal_seascape_class'] == 23,
    '8_DAY_WINDOW'
]

print(windows_23)

## Merge monthly and 8-day seascape tables into a single file. Monthly and 8-day cells must be run prior to this merge

In [ ]:
# Start with the 8-day data as the base
merged_df = conc_class_per_site_8day.copy()

# Directly append the two columns from the monthly data
# This assumes both tables have the exact same row order
merged_df['modal_class_monthly'] = conc_class_per_site_monthly['modal_seascape_class']
merged_df['mean_prob_monthly'] = conc_class_per_site_monthly['mean_seascape_prob']

# remane the columns for 8-day seascapes
merged_df.rename(columns={'modal_seascape_class': 'modal_class_8day'}, inplace=True)  
merged_df.rename(columns={'mean_seascape_prob': 'mean_prob_8day'}, inplace=True)

# --- Save the Final Merged File ---
output_filename = "seascape_argo_summary_appended.tsv"
merged_df.to_csv(output_filename, sep='\t', index=False)

print("\n--- Appending Complete ---")
print("Final appended summary:")
print(merged_df.head())
print(f"\nResults saved to '{output_filename}'")

### Map Argo records color coded by seascape

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import requests
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# --- 1. Define Domain Limits (From previous cells) ---
n_lat = 32
s_lat = 10
e_lon = -59
w_lon = -100

# --- 2. Load the Monthly Summary Data ---
df_argo = pd.read_csv('seascape_argo_summary_monthly.tsv', sep='\t')

# --- 3. Download Bathymetry Data (ETOPO) ---
topo_filename = 'topo_data.nc'
parent_topo_filename = os.path.join('..', topo_filename)

topo_url = (
    f"https://coastwatch.pfeg.noaa.gov/erddap/griddap/etopo180.nc?altitude"
    f"[({min(s_lat, n_lat)}):1:({max(s_lat, n_lat)})]"
    f"[({min(w_lon, e_lon)}):1:({max(w_lon, e_lon)})]"
)

# Check if the file already exists to avoid re-downloading
if os.path.exists(topo_filename):
    print(f"Found local file {topo_filename}. Skipping download.")
elif os.path.exists(parent_topo_filename):
    topo_filename = parent_topo_filename
    print(f"Found local file one directory up: {topo_filename}. Skipping download.")
else:
    print(f"Attempting to download bathymetry from: {topo_url}")
    try:
        r = requests.get(topo_url, timeout=120)
        if r.status_code == 200:
            with open(topo_filename, 'wb') as f:
                f.write(r.content)
            print("Bathymetry download complete.")
        else:
            print(f"Download failed with status {r.status_code}. Using fallback method...")
            topo_ds = None
    except Exception as e:
        print(f"Error occurred: {e}")
        topo_ds = None

# Subset bathymetry to our exact Gulf of Mexico bounding box
# Open topo file and subset to our bounding box, robust to variable/coord names
try:
    topo_ds = xr.open_dataset(topo_filename)
except Exception as e:
    print(f"Unable to open {topo_filename}: {e}")
    topo_ds = None

if topo_ds is None:
    bathy_sub = None
else:
    # Pick a sensible data variable (common names: 'bed','altitude','z',...)
    candidates = ['bed', 'altitude', 'elevation', 'z', 'depth', 'alt']
    var = next((v for v in candidates if v in topo_ds.data_vars), None)
    if var is None:
        var = list(topo_ds.data_vars)[0]  # fallback

    # Find latitude/longitude coordinate names
    lat_candidates = ['latitude', 'lat', 'y']
    lon_candidates = ['longitude', 'lon', 'x']
    lat_name = next((n for n in lat_candidates if n in topo_ds.coords), None)
    lon_name = next((n for n in lon_candidates if n in topo_ds.coords), None)
    if lat_name is None or lon_name is None:
        raise RuntimeError(f"Latitude/longitude coords not found in {topo_filename}. Available coords: {list(topo_ds.coords)}")

    # Subset and rename coords to match downstream code expecting 'latitude'/'longitude'
    bathy = topo_ds[var].sel({lat_name: slice(s_lat, n_lat), lon_name: slice(w_lon, e_lon)})
    if lat_name != 'latitude' or lon_name != 'longitude':
        bathy = bathy.rename({lat_name: 'latitude', lon_name: 'longitude'})

    bathy_sub = bathy

# --- 4. Set Up Dynamic Colors for Seascape Classes ---
# Find every unique class across the entire dataset to ensure absolute consistency
# (Assumes 'ds' is already loaded in memory from previous cells)
all_values = ds['CLASS'].values
global_classes = np.unique(all_values[~np.isnan(all_values)]).astype(int)
num_global_classes = len(global_classes)

# Select a base color palette. 
if num_global_classes <= 20:
    colors = plt.get_cmap('tab20b')(np.linspace(0, 1, num_global_classes))
else:
    combined_palette = np.vstack([
        plt.get_cmap('tab20b')(np.linspace(0, 1, 20)),
        plt.get_cmap('tab20c')(np.linspace(0, 1, 20))
    ])
    colors = combined_palette[:num_global_classes]

# Create a dictionary for easy color lookup
seascape_palette = {cls: mcolors.to_hex(colors[i]) for i, cls in enumerate(global_classes)}

# Drop NaN seascape values so they don't plot as black points
df_argo_clean = df_argo.dropna(subset=['modal_seascape_class']).copy()

# Map the colors to a new column. Any unlisted classes will default to black
df_argo_clean['color'] = df_argo_clean['modal_seascape_class'].map(seascape_palette).fillna('black')

# --- 5. Create the Map ---
fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())

# --- 6. Plot Bathymetry Contours ---
# Create filled contours from -5000m up to 0m using a reversed blue-purple colormap
if bathy_sub is not None:
    levels = np.arange(-5000, 1, 500)
    bathy_contour = ax.contourf(
        bathy_sub['longitude'], 
        bathy_sub['latitude'], 
        bathy_sub.values,
        levels=levels,
        cmap='PuBu_r', 
        extend='min',
        transform=ccrs.PlateCarree(),
        zorder=0
    )

    # Highlight the isobath with a black line
    ax.contour(
        bathy_sub['longitude'], 
        bathy_sub['latitude'], 
        bathy_sub.values,
        levels=[-1500, 0],
        colors='black',
        linewidths=1,
        transform=ccrs.PlateCarree(),
        zorder=1
    )

ax.add_feature(cfeature.LAND, zorder=2, edgecolor='none', facecolor='#A9A9A9') # Gray landmass
ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5, zorder=3)

# --- 7. Plot Argo Locations ---
scatter = ax.scatter(
    df_argo_clean['LONGITUDE'], 
    df_argo_clean['LATITUDE'], 
    c=df_argo_clean['color'], 
    s=45, 
    edgecolors='none', 
    transform=ccrs.PlateCarree(),
    zorder=4
)

# --- 8. Build Legends & Colorbars ---
# Custom Seascape Class Legend
# Filter to show only the classes actually present in the plotted Argo records
unique_argo_classes = np.sort(df_argo_clean['modal_seascape_class'].unique()).astype(int)
legend_patches = [
    mpatches.Circle((0, 0), radius=5, color=seascape_palette[cls], label=str(cls)) 
    for cls in unique_argo_classes if cls in seascape_palette
]

# Push legend to the top-right corner to avoid colorbar collision
leg = ax.legend(
    handles=legend_patches, 
    title='Seascape Class', 
    loc='upper left', 
    bbox_to_anchor=(1.1, 1.1), 
    fontsize=8, 
    title_fontsize=8,
    framealpha=1,
    edgecolor='black',
    borderpad=1,
    labelspacing=1.2,
    ncol=2 if len(unique_argo_classes) > 10 else 1 
)

# Dynamically anchor the colorbar to the bottom-right half of the axes
if bathy_sub is not None:
    cbar = plt.colorbar(bathy_contour, ax=ax, orientation='vertical', shrink=0.5, pad=0.1, anchor=(0, 0))
    cbar.set_label('Bathymetry (m)', fontsize=8, weight='bold')
    cbar.set_ticks([-5000, -4000, -3000, -2000, -1000, 0])

ax.set_title('Argo Profiles by Seascape Class (monthly) with Bathymetry', fontsize=16, weight='bold', pad=15)
plt.show()

### Define Quadrants & Map Selected Date using monthly seascapes

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- 0. Configuration & File Selection ---
# Set to 'monthly' or '8day'
DATASET_TYPE = 'monthly'

# Target date string (nearest available timestamp will be selected)
TARGET_DATE = '2024-12-15'

nc_files = {
    'monthly': 'seascape_argo_data_seascape_month.nc',
    '8day': 'seascape_argo_data_seascape_8day.nc'
}

# Open selected NetCDF dataset
ds = xr.open_dataset(nc_files[DATASET_TYPE])

# Load Argo data to overlay
argo_df = pd.read_csv('argo_sprof_processed_data.csv')
argo_df['datetime'] = pd.to_datetime(argo_df['datetime'])

# --- 1. Align Bathymetry to Seascape Grid ---
print(f"Processing {DATASET_TYPE} dataset. Interpolating bathymetry and constructing spatial masks...")
bathy_interp = bathy_sub.interp(latitude=ds.lat, longitude=ds.lon, method='nearest')
lon_grid, lat_grid = np.meshgrid(ds.lon.values, ds.lat.values)

# --- 2. Define Region Masks ---
# Common Depth Condition: >1000m (values <= -1000)
depth_mask = bathy_interp.values <= -1000

# Gulf of Mexico Mask (Isolated GoM domain)
gom_mask = (lat_grid >= 18.0) & (lat_grid <= 31.0) & (lon_grid >= -98.0) & (lon_grid <= -80.0)
gom_mask &= ~((lat_grid < 22.0) & (lon_grid > -88.0)) # Cut off Yucatan Channel / Caribbean
gom_mask &= ~(lon_grid > -80.0) # Cut off Straits of Florida

# Caribbean Sea Mask (Caribbean domain)
carib_mask = (lat_grid >= 8.0) & (lat_grid <= 20.0) & (lon_grid >= -75.0) & (lon_grid <= -60.0)
carib_mask &= ~((lat_grid > 21.0) & (lon_grid < -86.0)) # Exclude GoM spillover

# Final Combined Masks (Both restricted to depth > 1000m)
valid_mask_gom = depth_mask & gom_mask
valid_mask_carib = depth_mask & carib_mask

# --- 3. Define GoM Quadrants ---
lat_c = 25.78
lon_c = -89.86

quadrant_masks = {
    '1-NE': valid_mask_gom & (lat_grid >= lat_c) & (lon_grid >= lon_c),
    '2-NW': valid_mask_gom & (lat_grid >= lat_c) & (lon_grid < lon_c),
    '3-SE': valid_mask_gom & (lat_grid < lat_c) & (lon_grid >= lon_c),
    '4-SW': valid_mask_gom & (lat_grid < lat_c) & (lon_grid < lon_c)
}

# --- 4. Define Dynamic Global Color Palette ---
all_values = ds['CLASS'].values
global_classes = np.unique(all_values[~np.isnan(all_values)]).astype(int)
num_global_classes = len(global_classes)

if num_global_classes <= 20:
    colors = plt.get_cmap('tab20b')(np.linspace(0, 1, num_global_classes))
else:
    combined_palette = np.vstack([
        plt.get_cmap('tab20b')(np.linspace(0, 1, 20)),
        plt.get_cmap('tab20c')(np.linspace(0, 1, 20))
    ])
    colors = combined_palette[:num_global_classes]

common_cmap = ListedColormap(colors)
common_bounds = np.append(global_classes, global_classes[-1] + 1) - 0.5
common_norm = BoundaryNorm(common_bounds, ncolors=common_cmap.N)
seascape_palette = {cls: mcolors.to_hex(colors[i]) for i, cls in enumerate(global_classes)}

# --- 5. Extract Nearest Time Slice ---
seascape_slice = ds['CLASS'].sel(time=TARGET_DATE, method='nearest')
if 'time' in seascape_slice.dims:
    seascape_slice = seascape_slice.isel(time=0)

time_fmt = '%Y-%m-%d' if DATASET_TYPE == '8day' else '%Y-%m'
slice_date = pd.to_datetime(seascape_slice.time.values)
slice_time_str = slice_date.strftime(time_fmt)

# Filter Argo Data to Match the Seascape Time Slice
if DATASET_TYPE == 'monthly':
    argo_subset = argo_df[(argo_df['datetime'].dt.year == slice_date.year) & 
                          (argo_df['datetime'].dt.month == slice_date.month)]
else:
    end_date = slice_date + pd.Timedelta(days=8)
    argo_subset = argo_df[(argo_df['datetime'] >= slice_date) & 
                          (argo_df['datetime'] < end_date)]

# --- 6. Plotting Helper Function ---
def plot_domain(map_mask, extent, title, show_quadrants=False, isobath=None, argo_data=None):
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    map_data = seascape_slice.values.copy()
    map_data[~map_mask] = np.nan

    # Plot Seascapes
    mesh = ax.pcolormesh(
        ds.lon.values,
        ds.lat.values,
        map_data,
        cmap=common_cmap,
        norm=common_norm,
        transform=ccrs.PlateCarree(),
        zorder=2
    )

    # Contour Isobath
    if isobath is not None:
        ax.contour(
            bathy_sub['longitude'],
            bathy_sub['latitude'],
            bathy_sub.values,
            levels=[isobath],
            colors='black',
            linewidths=1.5,
            linestyles='solid',
            transform=ccrs.PlateCarree(),
            zorder=4
        )

    # Plot Quadrants (for GoM)
    if show_quadrants:
        lat_idx = np.abs(ds.lat.values - lat_c).argmin()
        valid_lons = ds.lon.values[valid_mask_gom[lat_idx, :]]
        if len(valid_lons) > 0:
            ax.plot([valid_lons.min(), valid_lons.max()], [lat_c, lat_c], color='black', linewidth=1.5, linestyle='-', zorder=4, transform=ccrs.PlateCarree())

        lon_idx = np.abs(ds.lon.values - lon_c).argmin()
        valid_lats = ds.lat.values[valid_mask_gom[:, lon_idx]]
        if len(valid_lats) > 0:
            ax.plot([lon_c, lon_c], [valid_lats.min(), valid_lats.max()], color='black', linewidth=1.5, linestyle='-', zorder=4, transform=ccrs.PlateCarree())

        ax.text(-86.5, 28.2, '1-NE', fontsize=16, weight='bold', transform=ccrs.PlateCarree(), zorder=5)
        ax.text(-93.0, 28.2, '2-NW', fontsize=16, weight='bold', transform=ccrs.PlateCarree(), zorder=5)
        ax.text(-86.5, 25.0, '3-SE', fontsize=16, weight='bold', transform=ccrs.PlateCarree(), zorder=5)
        ax.text(-93.0, 25.0, '4-SW', fontsize=16, weight='bold', transform=ccrs.PlateCarree(), zorder=5)

    # Add Map Features
    ax.add_feature(cfeature.LAND, zorder=3, edgecolor='black', facecolor='#d3d3d3')
    ax.add_feature(cfeature.COASTLINE, zorder=3)
    ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5, zorder=1)

    # Plot Argo Occurrences
    argo_legend_element = None
    if argo_data is not None and not argo_data.empty:
        # Limit to extents to avoid rendering out-of-bounds data
        lon_min, lon_max, lat_min, lat_max = extent
        domain_argo = argo_data[
            (argo_data['LONGITUDE'] >= lon_min) & (argo_data['LONGITUDE'] <= lon_max) &
            (argo_data['LATITUDE'] >= lat_min) & (argo_data['LATITUDE'] <= lat_max)
        ]
        
        if not domain_argo.empty:
            ax.scatter(
                domain_argo['LONGITUDE'], 
                domain_argo['LATITUDE'],
                s=120, c='red', marker='*', edgecolor='black', linewidth=0.5,
                transform=ccrs.PlateCarree(), zorder=6
            )
            argo_legend_element = Line2D([0], [0], marker='*', color='w', markerfacecolor='red', 
                                         markeredgecolor='black', markersize=12, label='Argo Float')

    # Legend for active classes in map slice
    unique_in_slice = np.unique(map_data[~np.isnan(map_data)]).astype(int)
    legend_patches = [mpatches.Circle((0, 0), radius=5, color=seascape_palette[cls], label=str(int(cls))) for cls in unique_in_slice]
    
    # Append Argo symbol to legend if active
    if argo_legend_element:
        legend_patches.append(argo_legend_element)
        
    ax.legend(handles=legend_patches, title='Map Features', loc='lower right', fontsize=10, title_fontsize=11, framealpha=1, edgecolor='black', ncol=2)

    ax.set_title(f"{title}\n{DATASET_TYPE.capitalize()} Seascapes for {slice_time_str}", fontsize=15, weight='bold', pad=15)
    plt.show()

# --- 7. Render Maps ---
# Map 1: Gulf of Mexico
plot_domain(
    map_mask=valid_mask_gom,
    extent=[-98, -80, 18, 31],
    title="Target GoM ROI & Quadrants (>1000m Isobath)",
    show_quadrants=True,
    isobath=-1000,
    argo_data=argo_subset
)

# Map 2: Caribbean Sea Domain (>1000m Isobath)
plot_domain(
    map_mask=valid_mask_carib,
    extent=[-89, -60, 8, 23],
    title="Caribbean Sea Domain (>1000m Isobath)",
    show_quadrants=False,
    isobath=-1000,
    argo_data=argo_subset
)

### Extract Seascape Time Series & Plot Stacked Area Charts

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.lines import Line2D  # Used to append the custom red dashed line to legends

# --- 0. Set Dataset Type & Prepare Regions ---
# Options: 'monthly' or '8day'
DATASET_TYPE = "monthly"

nc_files = {
    "monthly": "seascape_argo_data_seascape_month.nc",
    "8day": "seascape_argo_data_seascape_8day.nc",
}

# Open selected NetCDF dataset if not already loaded
ds = xr.open_dataset(nc_files[DATASET_TYPE])

# Load Argo data to compute occurrences
# (Assuming argo_sprof_processed_data.csv is available from previous steps)
argo_df = pd.read_csv('argo_sprof_processed_data.csv')
argo_df['datetime'] = pd.to_datetime(argo_df['datetime'])

# Filter data to count ONLY unique Argo profiles (1 row per WMOID + CYCLE_NUMBER)
unique_argo_df = argo_df.drop_duplicates(subset=['WMOID', 'CYCLE_NUMBER']).copy()

# Define time axis and dynamic label
times = pd.to_datetime(ds.time.values)
freq_label = "8-Day" if DATASET_TYPE == "8day" else "Monthly"

# Combine GoM quadrants and Caribbean region into a single processing dictionary
# Note: quadrant_masks, valid_mask_carib, global_classes, and colors must be defined in earlier cells
all_regions = {**quadrant_masks, "Caribbean": valid_mask_carib}

# --- 1. Process Argo Occurrences ---
print(f"Calculating Argo occurrences per region for {freq_label} intervals...")
lat_c = 25.78
lon_c = -89.86

def assign_region(lat, lon):
    # Base masks to replicate domain bounding boxes
    is_gom = (18.0 <= lat <= 31.0) and (-98.0 <= lon <= -80.0)
    is_gom &= not ((lat < 22.0) and (lon > -88.0))
    is_gom &= not (lon > -80.0)
    
    is_carib = (8.0 <= lat <= 20.0) and (-75.0 <= lon <= -60.0)
    is_carib &= not ((lat > 21.0) and (lon < -86.0))
    
    if is_gom:
        if lat >= lat_c and lon >= lon_c: return '1-NE'
        elif lat >= lat_c and lon < lon_c: return '2-NW'
        elif lat < lat_c and lon >= lon_c: return '3-SE'
        else: return '4-SW'
    elif is_carib:
        return 'Caribbean'
    return None

unique_argo_df['region'] = unique_argo_df.apply(lambda row: assign_region(row['LATITUDE'], row['LONGITUDE']), axis=1)

# Create chronological bins to map Argo datetimes into the NetCDF time windows
if DATASET_TYPE == "8day":
    time_bins = list(times) + [times[-1] + pd.Timedelta(days=8)]
else:
    time_bins = list(times) + [times[-1] + pd.DateOffset(months=1)]

unique_argo_df['time_idx'] = pd.cut(unique_argo_df['datetime'], bins=time_bins, labels=False, right=False)

# Calculate occurrence counts per region and time bin
argo_counts = {r: np.zeros(len(times)) for r in all_regions}
for r in all_regions:
    counts = unique_argo_df[unique_argo_df['region'] == r].groupby('time_idx').size()
    for idx, count in counts.items():
        if pd.notna(idx):
            argo_counts[r][int(idx)] = count

# --- 2. Extract Seascape Time Series (>1000m Isobath) ---
print(f"Extracting {freq_label} seascape time series for GoM quadrants and Caribbean...")

ts_data = {
    r: pd.DataFrame(index=times, columns=global_classes).fillna(0.0)
    for r in all_regions
}

for i in range(len(times)):
    slice_class = ds.isel(time=i)["CLASS"].values

    for r, mask in all_regions.items():
        pixels = slice_class[mask]
        valid_pixels = pixels[~np.isnan(pixels)]
        total = len(valid_pixels)

        if total > 0:
            counts = pd.Series(valid_pixels).value_counts()
            for c in global_classes:
                ts_data[r].loc[times[i], c] = (counts.get(c, 0) / total) * 100

class_labels = [str(int(c)) for c in global_classes]

# --- 3. Plot 1: Gulf of Mexico Subregions (2x2 Grid) ---
fig_gom, axes_gom = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes_gom = axes_gom.flatten()

plot_order_gom = ["2-NW", "1-NE", "4-SW", "3-SE"]

# Establish a consistent secondary y-axis limit across all GoM subplots
max_argo_gom = max([argo_counts[q].max() for q in plot_order_gom])
max_argo_gom = max(max_argo_gom * 1.1, 5) # Buffer scaling, minimum ceiling of 5

for idx, q in enumerate(plot_order_gom):
    ax = axes_gom[idx]
    df_q = ts_data[q]

    # Primary Y-axis: Seascape Relative Extent
    ax.stackplot(
        df_q.index,
        df_q.T.values,
        labels=class_labels,
        colors=colors,
        alpha=0.95,
    )
    ax.set_title(f"Quadrant {q}", fontsize=16, weight="bold")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    if idx % 2 == 0:
        ax.set_ylabel(f"{freq_label} Relative Extent (%)", fontsize=12)

    # Secondary Y-axis: Argo Occurrences
    ax2 = ax.twinx()
    ax2.plot(df_q.index, argo_counts[q], color='red', linestyle='dashed', linewidth=1.5)
    ax2.set_ylim(0, max_argo_gom)
    ax2.tick_params(axis='y', colors='red')
    
    if idx % 2 != 0:
        ax2.set_ylabel('Argo Occurrences', color='red', fontsize=12)
    else:
        ax2.tick_params(labelright=False) # Hide counts inside center grid margin to avoid clutter

for ax in axes_gom[2:]:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", labelsize=12)

# Build legend for GoM active classes
handles_gom, leg_labels_gom = axes_gom[0].get_legend_handles_labels()
active_classes_gom = set()
for q in plot_order_gom:
    active_classes_gom.update(ts_data[q].loc[:, ts_data[q].max() > 0].columns)

filtered_handles_gom = [
    h for h, l in zip(handles_gom, leg_labels_gom) if int(l) in active_classes_gom
]
filtered_labels_gom = [l for l in leg_labels_gom if int(l) in active_classes_gom]

# Append custom handle for Argo Line to legend
filtered_handles_gom.append(Line2D([0], [0], color='red', linestyle='dashed', linewidth=2))
filtered_labels_gom.append('Argo Profiles')

fig_gom.legend(
    filtered_handles_gom,
    filtered_labels_gom,
    title="Map Features",
    loc="center right",
    bbox_to_anchor=(1.15, 0.5), # Shifted slightly right so it doesn't overlap the new Y-axis limits
    fontsize=11,
    title_fontsize=13,
    framealpha=1,
    edgecolor="black",
    ncol=1 if len(filtered_labels_gom) <= 12 else 2,
)

fig_gom.suptitle(
    f"{freq_label} Relative Extent of Seascapes in Deep Gulf of Mexico (>1000m Isobath)",
    fontsize=18,
    weight="bold",
    y=0.96,
)
fig_gom.tight_layout(rect=[0, 0, 1.0, 0.95])
plt.show()

# --- 4. Plot 2: Caribbean Sea Region ---
fig_carib, ax_carib = plt.subplots(figsize=(14, 6))

df_carib = ts_data["Caribbean"]
ax_carib.stackplot(
    df_carib.index,
    df_carib.T.values,
    labels=class_labels,
    colors=colors,
    alpha=0.95,
)

ax_carib.set_title(
    f"{freq_label} Relative Extent of Seascapes in Deep Caribbean Sea (>1000m Isobath)",
    fontsize=16,
    weight="bold",
    pad=15,
)
ax_carib.set_ylim(0, 100)
ax_carib.set_ylabel(f"{freq_label} Relative Extent (%)", fontsize=12)
ax_carib.grid(axis="y", linestyle="--", alpha=0.4)

ax_carib.xaxis.set_major_locator(mdates.YearLocator())
ax_carib.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_carib.tick_params(axis="x", labelsize=12)

# Secondary Y-axis for Caribbean Argo Occurrences
ax2_carib = ax_carib.twinx()
ax2_carib.plot(df_carib.index, argo_counts["Caribbean"], color='red', linestyle='dashed', linewidth=1.5)
ax2_carib.set_ylabel('Argo Occurrences', color='red', fontsize=12)
ax2_carib.tick_params(axis='y', colors='red')
ax2_carib.set_ylim(0, max(argo_counts["Caribbean"].max() * 1.1, 5))

# Build legend for Caribbean active classes
active_classes_carib = set(
    df_carib.loc[:, df_carib.max() > 0].columns
)
filtered_handles_carib = [
    h
    for h, l in zip(handles_gom, leg_labels_gom) # Reusing handles generated from GoM stackplot
    if l.isdigit() and int(l) in active_classes_carib
]
filtered_labels_carib = [
    l for l in leg_labels_gom if l.isdigit() and int(l) in active_classes_carib
]

# Append custom handle for Argo line
filtered_handles_carib.append(Line2D([0], [0], color='red', linestyle='dashed', linewidth=2))
filtered_labels_carib.append('Argo Profiles')

fig_carib.legend(
    filtered_handles_carib,
    filtered_labels_carib,
    title="Map Features",
    loc="center right",
    bbox_to_anchor=(1.2, 0.5),
    fontsize=11,
    title_fontsize=13,
    framealpha=1,
    edgecolor="black",
    ncol=1 if len(filtered_labels_carib) <= 12 else 2,
)

fig_carib.tight_layout(rect=[0, 0, 1.0, 1.0])
plt.show()

## Isolate a selected seascape

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# --- 0. Configuration ---
TARGET_CLASS = 9
REGION = "Caribbean"

fig_single, ax_single = plt.subplots(figsize=(14, 6))
df_target = ts_data[REGION][TARGET_CLASS]

# --- 1. Plot Isolated Class Frequency ---
# Plot line and fill underneath for the specific class
color_target = seascape_palette.get(TARGET_CLASS, 'blue')
ax_single.plot(df_target.index, df_target.values, color=color_target, linewidth=2, label=f"Class {TARGET_CLASS}")
ax_single.fill_between(df_target.index, 0, df_target.values, color=color_target, alpha=0.4)

ax_single.set_title(f"{freq_label} Relative Extent of Seascape Class {TARGET_CLASS}\nDeep {REGION} (>1000m Isobath)", fontsize=16, weight="bold", pad=15)
ax_single.set_ylim(0, max(df_target.max() * 1.2, 10))
ax_single.set_ylabel(f"{freq_label} Relative Extent (%)", fontsize=12)
ax_single.grid(axis="y", linestyle="--", alpha=0.4)

# Format X-axis dates
ax_single.xaxis.set_major_locator(mdates.YearLocator())
ax_single.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_single.tick_params(axis="x", labelsize=12)

# --- 2. Secondary Y-axis for Argo Occurrences ---
ax2_single = ax_single.twinx()
ax2_single.plot(df_target.index, argo_counts[REGION], color='red', linestyle='dashed', linewidth=1.5, label='Argo Profiles')
ax2_single.set_ylabel('Argo Occurrences', color='red', fontsize=12)
ax2_single.tick_params(axis='y', colors='red')
ax2_single.set_ylim(0, max(argo_counts[REGION].max() * 1.1, 5))

# --- 3. Combined Legend ---
lines_1, labels_1 = ax_single.get_legend_handles_labels()
lines_2, labels_2 = ax2_single.get_legend_handles_labels()
ax_single.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right", fontsize=11, framealpha=1, edgecolor="black")

fig_single.tight_layout()
plt.show()

## Render seascape frequency histograms

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 0. Configuration ---
# Define which seascape class column to analyze ('modal_class_8day' or 'modal_class_monthly')
CLASS_COL = 'modal_class_monthly'

# --- 1. Load and Filter Data ---
# Load the appended TSV file
df = pd.read_csv('seascape_argo_summary_appended.tsv', sep='\t')

# Ensure datetime is properly formatted
df['datetime'] = pd.to_datetime(df['datetime'])

# Drop duplicates to keep ONLY unique Argo profiles (1 row per WMOID + CYCLE_NUMBER)
unique_argo_df = df.drop_duplicates(subset=['WMOID', 'CYCLE_NUMBER']).copy()

# --- 2. Assign Regions ---
lat_c = 25.78
lon_c = -89.86

def assign_region(lat, lon):
    # Base masks to replicate domain bounding boxes
    is_gom = (18.0 <= lat <= 31.0) and (-98.0 <= lon <= -80.0)
    is_gom &= not ((lat < 22.0) and (lon > -88.0))
    is_gom &= not (lon > -80.0)
    
    is_carib = (8.0 <= lat <= 23.0) and (-89.0 <= lon <= -60.0)
    is_carib &= not ((lat > 21.0) and (lon < -86.0))
    
    if is_gom:
        if lat >= lat_c and lon >= lon_c: return '1-NE'
        elif lat >= lat_c and lon < lon_c: return '2-NW'
        elif lat < lat_c and lon >= lon_c: return '3-SE'
        else: return '4-SW'
    elif is_carib:
        return 'Caribbean'
    return None

# Apply the region assignment and filter out any points outside the domains or missing class data
unique_argo_df['region'] = unique_argo_df.apply(lambda row: assign_region(row['LATITUDE'], row['LONGITUDE']), axis=1)
unique_argo_df = unique_argo_df.dropna(subset=['region', CLASS_COL])

# --- 3. Plot Histograms ---
regions = ["2-NW", "1-NE", "4-SW", "3-SE", "Caribbean"]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, r in enumerate(regions):
    ax = axes[idx]
    region_data = unique_argo_df[unique_argo_df['region'] == r]
    
    # Calculate frequencies of each seascape class
    counts = region_data[CLASS_COL].value_counts().sort_index()
    
    if not counts.empty:
        # Convert index to string to treat classes as categorical variables on the x-axis
        x_labels = counts.index.astype(int).astype(str)
        bars = ax.bar(x_labels, counts.values, edgecolor='black', alpha=0.85)
        
        # Try to color the bars using the seascape_palette from earlier cells
        try:
            for bar, cls in zip(bars, counts.index):
                bar.set_facecolor(seascape_palette[int(cls)])
        except NameError:
            pass # Fallback to default colors if seascape_palette is not in memory
            
    ax.set_title(f"{r} Region (n={len(region_data)} profiles)", fontsize=14, weight='bold')
    ax.set_xlabel('Seascape Class', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

# Hide the 6th (empty) subplot in the 3x2 grid
axes[5].set_visible(False)

fig.suptitle(f"Frequency Distribution of Modal Seascape Classes\n({CLASS_COL})", fontsize=18, weight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 0. Configuration ---
TARGET_REGION = 'Caribbean' # Change to '1-NE', '2-NW', '3-SE', '4-SW', or 'Caribbean'
TARGET_PARAM = 'salinity' # Options: 'temperature', 'salinity', 'doxy', 'chla', 'nitrate', 'ph', 'bbp700'
CLASS_COL = 'modal_class_8day' # Options: 'modal_class_8day' or 'modal_class_monthly'

MAX_DEPTH = 200      # Maximum depth (pressure) to plot
DEPTH_BIN_SIZE = 5   # Bin size in meters/dbars for averaging
MIN_SAMPLES = 3      # Minimum number of data points in a depth bin to plot

# Define axis labels for formatting
param_labels = {
    'temperature': 'Temperature (°C)',
    'salinity': 'Salinity',
    'doxy': 'Dissolved Oxygen (µmol/kg)',
    'chla': 'Chlorophyll-a (mg/m³)',
    'nitrate': 'Nitrate (µmol/kg)',
    'ph': 'pH (In-situ)',
    'bbp700': 'Backscattering at 700nm (m⁻¹)'
}

# --- 1. Load and Pre-Filter Data ---
print(f"Loading data and processing vertical profiles for {TARGET_PARAM} in {TARGET_REGION}...")

# Load depth-resolved Argo profile data and the appended seascape summary
profiles_df = pd.read_csv('argo_sprof_processed_data.csv')
seascape_df = pd.read_csv('seascape_argo_summary_appended.tsv', sep='\t')

# OPTIMIZATION: Filter profiles early to drop deep measurements and NaN values to drastically speed up processing
valid_profiles = profiles_df[
    (profiles_df['pressure'] <= MAX_DEPTH) & 
    (profiles_df[TARGET_PARAM].notna())
].copy()

# --- 2. Vectorized Region Assignment ---
lat_c, lon_c = 25.78, -89.86

lat = valid_profiles['LATITUDE']
lon = valid_profiles['LONGITUDE']

# Define geographic masks using vectorized boolean logic
is_gom = (lat >= 18.0) & (lat <= 31.0) & (lon >= -98.0) & (lon <= -80.0) & ~((lat < 22.0) & (lon > -88.0)) & ~(lon > -80.0)
is_carib = (lat >= 8.0) & (lat <= 20.0) & (lon >= -75.0) & (lon <= -60.0) & ~((lat > 21.0) & (lon < -86.0))

# Assign region names based on conditions
conditions = [
    is_gom & (lat >= lat_c) & (lon >= lon_c),
    is_gom & (lat >= lat_c) & (lon < lon_c),
    is_gom & (lat < lat_c) & (lon >= lon_c),
    is_gom & (lat < lat_c) & (lon < lon_c),
    is_carib
]
choices = ['1-NE', '2-NW', '3-SE', '4-SW', 'Caribbean']

# OPTIMIZATION: Instant assignment without iterating through rows
valid_profiles['region'] = np.select(conditions, choices, default=None)

# OPTIMIZATION: Filter to just the target region *before* the merge
region_profiles = valid_profiles[valid_profiles['region'] == TARGET_REGION]

# Merge to assign the regional seascape class to each depth-level measurement
region_df = pd.merge(
    region_profiles,
    seascape_df[['WMOID', 'CYCLE_NUMBER', CLASS_COL]],
    on=['WMOID', 'CYCLE_NUMBER'],
    how='inner'
)

# Keep only rows with valid seascape classes
region_df = region_df[region_df[CLASS_COL].notna()].copy()

# --- 3. Bin Depths and Calculate Statistics ---
# Create regular pressure bins to aggregate profiles
bins = np.arange(0, MAX_DEPTH + DEPTH_BIN_SIZE, DEPTH_BIN_SIZE)
midpoints = bins[:-1] + (DEPTH_BIN_SIZE / 2.0) 

region_df['depth_mid'] = pd.cut(region_df['pressure'], bins=bins, labels=midpoints).astype(float)

# CHANGE: Use 'std' (Standard Deviation) instead of 'sem' to show the data's natural envelope
profile_stats = region_df.groupby([CLASS_COL, 'depth_mid'])[TARGET_PARAM].agg(['mean', 'std', 'count']).reset_index()

# Filter out bins with insufficient data to prevent noisy statistical spikes
profile_stats = profile_stats[profile_stats['count'] >= MIN_SAMPLES]

# --- 4. Render Vertical Profile Plot ---
fig, ax = plt.subplots(figsize=(8, 10))

active_classes = np.sort(profile_stats[CLASS_COL].unique())

for cls in active_classes:
    cls_data = profile_stats[profile_stats[CLASS_COL] == cls].sort_values('depth_mid')
    if cls_data.empty:
        continue
        
    try:
        color = seascape_palette[int(cls)]
    except NameError:
        cmap = plt.get_cmap('tab20')
        color = cmap((int(cls) % 20) / 20.0)
        
    # Plot Mean Line
    ax.plot(cls_data['mean'], cls_data['depth_mid'], color=color, linewidth=2.5, label=f"Class {int(cls)}")
    
    # CHANGE: Plot Standard Deviation Envelope
    ax.fill_betweenx(
        cls_data['depth_mid'], 
        cls_data['mean'] - cls_data['std'], 
        cls_data['mean'] + cls_data['std'], 
        color=color, 
        alpha=0.2,            # Lowered alpha slightly for better overlapping visibility
        edgecolor='none'      # Removes the border on the fill
    )

ax.invert_yaxis() # Depth increases downwards
ax.set_title(f"Parameter: {TARGET_PARAM.capitalize()}\n{TARGET_REGION} Domain by Seascape Class", fontsize=15, weight='bold', pad=15)
ax.set_xlabel(param_labels.get(TARGET_PARAM, TARGET_PARAM), fontsize=12, weight='bold')
ax.set_ylabel('Pressure (dbar)', fontsize=12, weight='bold')

# --- Move X-Axis to Top ---
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(title='Seascape Class', loc='lower right', fontsize=11, framealpha=0.9, edgecolor='black')

fig.tight_layout()
plt.show()